In [2]:
import pyspark
import os
import sys
from pyspark.sql import SparkSession

# Notebook'un kullandığı Python yolunu al
python_path = sys.executable
print("Using Python:", python_path)

# Driver ve worker aynı Python'u kullansın
os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

# Eski session varsa kapat
try:
    spark.stop()
except:
    pass

# Yeni session başlat
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("test") \
    .config("spark.pyspark.python", python_path) \
    .config("spark.pyspark.driver.python", python_path) \
    .getOrCreate()

print("Spark started")

Using Python: /Users/selcukkaleli/data-engineering-zoomcamp/.venv/bin/python3


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 23:31:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark started


In [3]:
from pyspark.sql import types

In [4]:
df_nyc = spark.read.parquet('./data/homework_data/')

In [5]:
df_nyc = df_nyc.repartition(4)

In [7]:
df_nyc.write.mode("overwrite").parquet('./data/homework_data/question_2')

In [6]:
df_nyc = spark.read.parquet('./data/homework_data/question_2')

In [7]:
df_nyc.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [15]:
df_nyc.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2025-11-02 08:11:08|  2025-11-02 08:15:21|              1|         1.24|         1|                 N|         186|    

In [8]:
df_nyc.createOrReplaceTempView('trips_data_nov_25')

In [27]:
df_result = spark.sql("""
SELECT 
    COUNT(*)
FROM 
    trips_data_nov_25
WHERE
    tpep_pickup_datetime >= '2025-11-15 00:00:00' AND
    tpep_pickup_datetime < '2025-11-16 00:00:00'
""")

In [28]:
df_result.show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [13]:
df_result_4 = spark.sql("""
SELECT
    tpep_pickup_datetime,
    tpep_dropoff_datetime,
    (unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600.0 AS trip_hours
FROM trips_data_nov_25
ORDER BY trip_hours DESC
LIMIT 10
""")

In [14]:
df_result_4.show()

+--------------------+---------------------+----------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_hours|
+--------------------+---------------------+----------+
| 2025-11-26 20:22:12|  2025-11-30 15:01:00| 90.646667|
| 2025-11-27 04:22:41|  2025-11-30 09:19:35| 76.948333|
| 2025-11-03 10:42:55|  2025-11-06 14:55:45| 76.213889|
| 2025-11-07 11:23:22|  2025-11-10 08:40:41| 69.288611|
| 2025-11-18 17:12:47|  2025-11-21 12:17:37| 67.080556|
| 2025-11-22 17:45:30|  2025-11-25 09:07:36| 63.368333|
| 2025-11-01 07:44:57|  2025-11-03 16:07:53| 56.382222|
| 2025-11-27 14:34:56|  2025-11-29 14:43:48| 48.147778|
| 2025-11-01 14:55:34|  2025-11-03 14:24:16| 47.478333|
| 2025-11-22 16:05:20|  2025-11-24 13:31:59| 45.444167|
+--------------------+---------------------+----------+



In [15]:
df_zones = spark.read \
    .option("header", "true") \
    .csv('data/homework_data/taxi_zone_lookup.csv')

In [16]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [17]:
df_zones.createOrReplaceTempView('taxi_zones')

In [18]:
df_result_6 = df_nyc.join(df_zones, df_nyc.PULocationID == df_zones.LocationID)

In [20]:
df_result_6.createOrReplaceTempView('trips_data_zones')

In [30]:
df_result_6_answer = spark.sql("""
SELECT
    PULocationID,
    Zone,
    COUNT(*) AS number_of_visitor
FROM 
    trips_data_zones
GROUP BY
    PULocationID,
    Zone
ORDER BY
    number_of_visitor ASC
""")

In [31]:
df_result_6_answer.show()

+------------+--------------------+-----------------+
|PULocationID|                Zone|number_of_visitor|
+------------+--------------------+-----------------+
|          84|Eltingville/Annad...|                1|
|           5|       Arden Heights|                1|
|         105|Governor's Island...|                1|
|         187|       Port Richmond|                3|
|         199|       Rikers Island|                4|
|         111| Green-Wood Cemetery|                4|
|         204|   Rossville/Woodrow|                4|
|         109|         Great Kills|                4|
|           2|         Jamaica Bay|                5|
|         251|         Westerleigh|               12|
|         176|             Oakwood|               14|
|         245|       West Brighton|               14|
|         172|New Dorp/Midland ...|               14|
|          59|        Crotona Park|               14|
|         253|       Willets Point|               15|
|          27|Breezy Point/F